# Features

Run from the repo root, after `python -m similarity.preprocessing`.

In [ ]:
import numpy as np
import pandas as pd
import similarity.features as ft
from similarity.preprocessing import load_products

products = load_products()
f = ft.build_features(products)
len(f.ids)

## text

In [ ]:
pd.DataFrame({"name": products["name"].head(8), "prepared": products["name"].head(8).map(ft.prepare_text)})

In [ ]:
f.text.shape

In [ ]:
f.words[:30]

In [ ]:
(f.text.getnnz(axis=1) == 0).sum()

In [ ]:
products.loc[f.text.getnnz(axis=1) == 0, "name"]

In [ ]:
ft.top_words(f, 3)

In [ ]:
ft.top_words(f, 2)

In [ ]:
rows = [3, 1, 7, 2]
pd.DataFrame((f.text[rows] @ f.text[rows].T).toarray(), index=products.loc[rows, "name"].str[:45]).round(2)

In [ ]:
row = 3
scores = (f.text @ f.text[row].T).toarray().ravel()
top = np.argsort(scores)[::-1][:10]
products.loc[top, ["name", "child_category", "price"]].assign(text_score=scores[top].round(2))

## brand

In [ ]:
pd.DataFrame({"brand": products["brand"].head(10), "code": f.brand[:10]})

In [ ]:
(f.brand == -1).sum(), f.brand.max() + 1

## category

In [ ]:
pd.DataFrame({"category": products["child_category"].head(10), "code": f.category[:10]})

In [ ]:
(f.category == -1).sum(), f.category.max() + 1

In [ ]:
((f.category == -1) & (f.browsenode == -1)).sum()

## gender

In [ ]:
g = ft.genders(products)
g.value_counts(dropna=False)

In [ ]:
pd.DataFrame({"category": products["child_category"], "name": products["name"].str[:60], "gender": g}).sample(10, random_state=0)

In [ ]:
pd.crosstab(products["child_category"].notna(), g.notna())

In [ ]:
from_category = products["child_category"].map(lambda c: ft.derive_gender(c, ""))
from_title = products["name"].map(lambda n: ft.derive_gender(None, n))
both = from_category.notna() & from_title.notna()
pd.crosstab(from_category[both], from_title[both])

In [ ]:
(from_category[both] == from_title[both]).mean()

In [ ]:
(f.gender == -1).sum()

## colour

In [ ]:
colours = products["colours"].dropna()
changed = colours[colours.map(ft.normalise_colours) != colours]
pd.DataFrame({"before": changed, "after": changed.map(ft.normalise_colours)}).head(10)

In [ ]:
len(changed)

In [ ]:
f.colours.shape

In [ ]:
pd.Series(np.asarray(f.colours.sum(axis=0)).ravel(), index=f.colour_names).sort_values(ascending=False).head(15)

## price, weight, rating

In [ ]:
pd.DataFrame({"price": f.price, "weight": f.weight, "rating": f.rating}).describe()

## duplicates

In [ ]:
counts = np.bincount(f.asin)
(counts > 1).sum(), counts.max()